# News Headline Classification Training

This notebook trains a DistilBERT model to classify news headlines as **Fox News** or **NBC News**.

Author: Chih Yu Tsai, Aditya Pratap Singh, Xinjie Hu

Course: CIS 4190/5190 Applied Machine Learning Fall 2025

## Instructions:
1. GPU Usage: T4 GPU in colab
2. Data file: `85k_combined.xlsx` 
3. We split all the functions and run all cells in order
4. Download the generated `model.pt` after training completes


---
## Cell 1: Install Dependencies

In [ ]:
!pip install transformers pandas openpyxl scikit-learn tqdm -q
print("Dependencies installed!")

---
## Cell 2: Import Libraries & Check GPU

In [ ]:
import os
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import DistilBertTokenizer, DistilBertModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from urllib.parse import urlparse, unquote
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("=" * 50)
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected!")
    print("Change runtime type to GPU")
print("=" * 50)

---
## Cell 3: Configuration

Modify these settings if needed:

In [ ]:
class Config:
    """Training configuration - adjust these settings as needed."""
    
    # === DATA SETTINGS ===
    DATA_PATH = '85k_combined.xlsx'   # <-- our data file
    TEST_SIZE = 0.15                   # 15% for validation
    RANDOM_SEED = 42
    
    # === MODEL SETTINGS ===
    MODEL_NAME = 'distilbert-base-uncased'
    MAX_LENGTH = 128
    NUM_CLASSES = 2
    
    # === TRAINING SETTINGS ===
    BATCH_SIZE = 32                    # or 64
    LEARNING_RATE = 2e-5
    NUM_EPOCHS = 3
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01
    
    # === OUTPUT ===
    OUTPUT_PATH = 'model.pt'
    
    # === LABEL MAPPING ===
    LABEL_TO_ID = {'foxnews': 0, 'nbcnews': 1}
    ID_TO_LABEL = {0: 'foxnews', 1: 'nbcnews'}

print("✓ Configuration loaded!")
print(f"  Data file: {Config.DATA_PATH}")
print(f"  Batch size: {Config.BATCH_SIZE}")
print(f"  Epochs: {Config.NUM_EPOCHS}")

---
## Cell 4: URL Processing Functions

In [ ]:
def extract_label_from_url(url: str) -> str:
    """
    Extract news source label from URL domain.
    
    Examples:
        'https://www.foxnews.com/...' -> 'foxnews'
        'https://www.nbcnews.com/...' -> 'nbcnews'
    """
    url_lower = str(url).lower()
    if 'foxnews.com' in url_lower:
        return 'foxnews'
    elif 'nbcnews.com' in url_lower:
        return 'nbcnews'
    return None


def extract_text_from_url(url: str) -> str:
    """
    Extract readable headline text from URL path.
    
    Examples:
        'https://www.foxnews.com/world/australian-senator-wears-burqa'
        -> 'australian senator wears burqa'
    """
    try:
        parsed = urlparse(str(url))
        path = unquote(parsed.path)
        segments = [s for s in path.split('/') if s]
        
        if not segments:
            return ''
        
        # Find the best slug (mostly the last meaningful segment)
        slug = ''
        for segment in reversed(segments):
            # Skip ID-like segments (e.g., 'rcna240477')
            if re.match(r'^[a-z]*\d+$', segment.lower()):
                continue
            # Skip very short segments
            if len(segment) < 5:
                continue
            slug = segment
            break
        
        if not slug and segments:
            slug = segments[-1]
        
        # Clean the slug
        text = slug.replace('-', ' ').replace('_', ' ')
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip().lower()
        
        return text
    except:
        return ''


# Test the functions
test_url = 'https://www.foxnews.com/world/australian-senator-wears-burqa-after-move'
print("Testing URL extraction:")
print(f"  URL: {test_url}")
print(f"  Label: {extract_label_from_url(test_url)}")
print(f"  Text: {extract_text_from_url(test_url)}")
print("\n✓ URL processing functions ready!")

---
## Cell 5: Load and Preprocess Data

In [ ]:
def load_and_preprocess_data(data_path: str):
    """
    Load data file and extract texts/labels from URLs.
    """
    print(f"Loading data from: {data_path}")
    
    # Load file
    if data_path.endswith('.xlsx') or data_path.endswith('.xls'):
        df = pd.read_excel(data_path)
    else:
        df = pd.read_csv(data_path)
    
    print(f"Total rows: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    
    # Find URL column
    url_column = None
    for col in df.columns:
        if col.lower() in ['url', 'urls', 'link', 'links']:
            url_column = col
            break
    if url_column is None:
        url_column = df.columns[0]
    print(f"Using column: '{url_column}'")
    
    # Process URLs
    texts = []
    labels = []
    skipped = 0
    
    for url in tqdm(df[url_column], desc="Processing URLs"):
        if pd.isna(url):
            skipped += 1
            continue
        
        label = extract_label_from_url(url)
        if label is None:
            skipped += 1
            continue
        
        text = extract_text_from_url(url)
        if not text:
            skipped += 1
            continue
        
        texts.append(text)
        labels.append(label)
    
    print(f"\n✓ Successfully processed: {len(texts)}")
    print(f"✗ Skipped: {skipped}")
    print(f"\nLabel distribution:")
    print(f"  foxnews: {labels.count('foxnews')}")
    print(f"  nbcnews: {labels.count('nbcnews')}")
    
    return texts, labels


# Load the data
texts, labels = load_and_preprocess_data(Config.DATA_PATH)

---
## Cell 6: Split Data into Train/Validation

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(Config.RANDOM_SEED)
np.random.seed(Config.RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(Config.RANDOM_SEED)

# Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels,
    test_size=Config.TEST_SIZE,
    random_state=Config.RANDOM_SEED,
    stratify=labels  # Maintain label distribution
)

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")
print(f"\n✓ Data split complete!")

---
## Cell 7: Create PyTorch Dataset

In [ ]:
class NewsDataset(Dataset):
    """PyTorch Dataset for tokenized news headlines."""
    
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = [Config.LABEL_TO_ID[l] for l in labels]
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }


# Initialize tokenizer
print("Loading tokenizer...")
tokenizer = DistilBertTokenizer.from_pretrained(Config.MODEL_NAME)

# Create datasets
train_dataset = NewsDataset(train_texts, train_labels, tokenizer, Config.MAX_LENGTH)
val_dataset = NewsDataset(val_texts, val_labels, tokenizer, Config.MAX_LENGTH)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print("\n✓ Datasets created!")

---
## Cell 8: Define Model Architecture

In [ ]:
class NewsClassifier(nn.Module):
    """
    DistilBERT-based classifier for news headlines.
    
    Architecture:
        DistilBERT -> [CLS] token -> Dropout -> Linear(768, 2)
    """
    
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained(Config.MODEL_NAME)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(768, Config.NUM_CLASSES)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        cls_output = self.dropout(cls_output)
        return self.classifier(cls_output)


# Initialize model
print("Loading DistilBERT model...")
model = NewsClassifier().to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print("\n✓ Model initialized!")

---
## Cell 9: Setup Optimizer and Scheduler

In [ ]:
# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=Config.LEARNING_RATE,
    weight_decay=Config.WEIGHT_DECAY
)

# Learning rate scheduler
total_steps = len(train_loader) * Config.NUM_EPOCHS
warmup_steps = int(total_steps * Config.WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print("\n✓ Optimizer and scheduler ready!")

---
## Cell 10: Define Training Functions

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    
    pbar = tqdm(loader, desc="Training")
    for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = nn.CrossEntropyLoss()(logits, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(loader), accuracy_score(all_labels, all_preds)


def evaluate(model, loader, device):
    """Evaluate on validation set."""
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            logits = model(input_ids, attention_mask)
            loss = nn.CrossEntropyLoss()(logits, labels)
            
            total_loss += loss.item()
            all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return total_loss / len(loader), accuracy_score(all_labels, all_preds), all_preds, all_labels


print("✓ Training functions defined!")

---
## Cell 11: Train the Model

This cell will train the model for 3 epochs. 

In [ ]:
print("=" * 70)
print("STARTING TRAINING")
print("=" * 70)

best_acc = 0
best_epoch = 0

for epoch in range(Config.NUM_EPOCHS):
    print(f"\n{'='*70}")
    print(f"EPOCH {epoch + 1}/{Config.NUM_EPOCHS}")
    print('='*70)
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    
    # Evaluate
    val_loss, val_acc, _, _ = evaluate(model, val_loader, device)
    
    print(f"\n  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        best_epoch = epoch + 1
        torch.save(model.state_dict(), Config.OUTPUT_PATH)
        print(f"  ✓ Best model saved! (Acc: {val_acc:.4f})")

print("\n" + "=" * 70)
print("TRAINING COMPLETE!")
print("=" * 70)
print(f"\nBest Validation Accuracy: {best_acc:.4f} (Epoch {best_epoch})")

---
## Cell 12: Final Evaluation & Classification Report

In [ ]:
# Load best model
model.load_state_dict(torch.load(Config.OUTPUT_PATH))

# Final evaluation
_, final_acc, preds, true_labels = evaluate(model, val_loader, device)

print("\n" + "=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(true_labels, preds, target_names=['foxnews', 'nbcnews']))

# Show model file info
file_size = os.path.getsize(Config.OUTPUT_PATH) / (1024 * 1024)
print(f"\n✓ Model saved: {Config.OUTPUT_PATH} ({file_size:.1f} MB)")